# Act 1 — Day one of an AWS account

You just created an AWS account. You log in. You see a console full of services you have never touched, and one identity sitting at the top of everything you are about to do — the **root user**.

This act is about that identity and what you do with it on day one. Get this part wrong and nothing you build later is safe. Get it right and you can mostly forget it ever existed.

## The Root User

When you first create an AWS account, you get one identity for free — the **root user**. It owns the account and has unlimited, **un-restrictable** access. No policy can take permissions away from root.

### Tasks that *require* root
- Change account email or root password
- Close the account
- Restore admin access if you've locked yourself out
- Register as a seller in the Reserved Instance Marketplace
- A few billing tasks (e.g. enable IAM access to billing)

### Root hygiene — non-negotiable
- Long, unique password
- **Hardware MFA** enabled
- **No access keys ever issued for root**
- Credentials physically stored somewhere safe
- Then leave it alone — most accounts go months/years without anyone signing in as root

# Act 2 — Then the team grows

That first identity will not get you far. As soon as a second engineer joins, you have a real problem. Alice needs to read the analytics bucket. Bob needs to deploy Lambda functions. Carol needs to spin up dev environments. None of them should be able to close the account or change the billing email.

You need a way to express *who* is allowed to do *what* on *which* resource, under *which* conditions — and have AWS enforce it on every single API call. That mechanism is **IAM**, and it sits underneath every other AWS service.

This act introduces IAM — the kinds of identities you create, the shape of the policies that constrain them, and the five places those policies can live.

## What IAM is

**IAM** stands for **Identity and Access Management** — the AWS service that decides, for every API call, whether the caller is allowed to do what they're asking. Console click, CLI command, `boto3` call — they all funnel through the same authorization engine.

- **Global service** — no region selector. Users, groups, roles, and policies exist account-wide.
- **Default-deny** — if no policy explicitly allows an action, it is denied.
- **Free service** — you pay for what IAM grants access to, never for IAM itself.

## Three Identity Types

| Type | What it is | Credentials | Key rule |
|---|---|---|---|
| **User** | A single human or long-lived application | Console password and/or access key + secret | **One user per person** — never share |
| **Group** | A bag of users | None (groups don't authenticate) | No nesting; user can be in many groups |
| **Role** | An identity with policies but **no permanent credentials** | Temporary creds vended at assume time | The answer to most credential problems |

**Roles are the most important of the three.** Nobody logs in *as* a role — something *assumes* it, and at the moment of assumption AWS issues short-lived credentials that auto-expire. Roles are how:

- EC2 instances talk to S3 (instance profile)
- Lambda functions read from DynamoDB (execution role)
- Account A reaches a bucket in Account B (cross-account role)
- Federated employees enter AWS at all (SAML / OIDC)

**If you're creating long-lived access keys to solve a problem, the answer is almost always a role instead.**

## Role Assumption — Three Policies, Two Sides

Role assumption is a **two-sided handshake**. The role holds two policies, and the caller holds a third. All three must align — otherwise the `AssumeRole` call fails or the assumed session is useless.

| Side | Policy | Question it answers | Example |
|---|---|---|---|
| **Role** | **Trust policy** | *Who* is allowed to assume this role? | "the EC2 service" / "account 222222222222" / "this IAM user `inventory-backup-bot`" |
| **Role** | **Permissions policy** | *What* can the role do once assumed? | "read objects from this S3 bucket" |
| **Caller** | **Identity policy** | Is the caller *allowed to call* `sts:AssumeRole` on this role? | `Allow sts:AssumeRole on arn:aws:iam::111111111111:role/S3BackupRole` |

### Why the caller-side identity policy matters

Without it, AWS returns `AccessDenied` on the `sts:AssumeRole` call itself — before the trust policy is even evaluated. The trust policy on the role is necessary but not sufficient; the caller must also have permission to make the call.

### Concrete split — `inventory-backup-bot` assuming `S3BackupRole`

| Where | Policy attached | Contents |
|---|---|---|
| **On the bot (IAM user)** | Identity policy | `Allow sts:AssumeRole on arn:...:role/S3BackupRole` — and *nothing about S3* |
| **On the role** | Trust policy | `Principal: arn:...:user/inventory-backup-bot` allowed to `sts:AssumeRole` |
| **On the role** | Permissions policy | `Allow s3:GetObject, s3:PutObject on arn:aws:s3:::inventory-backups/*` |

The bot itself has **no S3 permissions**. It only has the right to *become* the role; the role is what carries S3 access. This is the whole point of role assumption — credentials with broad power exist only as short-lived sessions, never as long-lived keys on the bot.

> **Same-account vs cross-account:** the structure above is identical whether the bot and role live in the same account or different accounts. Cross-account just means the role's trust policy names a principal in a different account.

## Policies — Shape and Types

A policy is a JSON document. Every statement boils down to four fields:

| Field | What it means |
|---|---|
| **Effect** | `Allow` or `Deny` |
| **Action** | The API operation, e.g. `s3:GetObject`, `ec2:*` |
| **Resource** | The ARN the action applies to (or `*` for all) |
| **Condition** | *(optional)* Context checks (IP, MFA, region, tag, etc.) |

### Three managed flavors

| Type | Editable? | Reusable? | Use when |
|---|---|---|---|
| **AWS managed** | No | Yes | Common cases (e.g. `ReadOnlyAccess`, `AmazonS3FullAccess`) |
| **Customer managed** | Yes | Yes | Your reusable, version-controlled policies — **the default best practice** |
| **Inline** | Yes | No (one-to-one) | One-off, tightly bound to a single identity; deleted with the identity |

In [ ]:
# A minimal customer-managed policy: allow reading one bucket, but only over HTTPS.
import json

policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "ReadReportsBucket",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [
                "arn:aws:s3:::analytics-reports",
                "arn:aws:s3:::analytics-reports/*",
            ],
            "Condition": {
                "Bool": {"aws:SecureTransport": "true"}
            },
        }
    ],
}
print(json.dumps(policy, indent=2))

## The Five Policy Slots

Ordered by **scope** — broadest blast radius first, narrowest last. (This is the *organizational* order, not the *evaluation* order — see the next cell for how AWS actually combines them at request time.)

| # | Slot | Attaches to | Grants? | Caps? | What it does |
|---|---|---|---|---|---|
| 1 | **Service Control Policy (SCP)** | Org Root, OU, account | — | ✓ | Org-wide ceiling. Caps every principal in scope, **including the root user of member accounts**. The corporate law. |
| 2 | **Permission boundary** | User, role | — | ✓ | Per-identity ceiling. Caps what identity policies can ever grant — even if someone later attaches `AdministratorAccess`, the boundary still holds. |
| 3 | **Identity-based** | User, group, role | ✓ | — | The job description. **This is where most `Allow`s come from.** Without an `Allow` here (or in a resource-based policy), the answer is implicit deny. |
| 4 | **Resource-based** | Bucket, queue, topic, key, function | ✓ | — | A lock on the asset itself. **Special:** includes a `Principal` field, and enables cross-account access *without* requiring role assumption — Account B simply names Account A's principal in its bucket policy. |
| 5 | **Session policy** | Passed at `AssumeRole` time | — | ✓ | Per-session filter. Shrinks a role for one specific session; the next session of the same role is unaffected. |

### The single rule

When a call arrives, AWS checks all five together. The effective permission is the **intersection** — every cap (slots 1, 2, 5) must allow the action, *and* at least one grant (slot 3 or 4) must explicitly allow it. **An explicit `Deny` anywhere kills it, unconditionally.**

### The mental model — outermost ring inward

Slot 1 is set by the org. Slot 2 is set per identity. Slot 3 is the actual permission grant. Slot 4 lives on the resource being touched. Slot 5 is scoped to a single `AssumeRole` call. As you move down the list, the policy's blast radius shrinks and its lifetime gets shorter — from org-permanent at the top to single-session at the bottom.

### KMS Key Policies — A Special Case

Every KMS key **must** have a key policy, and unlike S3 buckets, the key policy is the **only** thing that grants access by default. IAM permissions on a user have no effect on a KMS key unless the key policy itself delegates authority to IAM.

**Why:** losing control of an IAM administrator account does not automatically mean losing control of your encryption keys.

# Act 3 — When AWS says yes or no

You now know all the places a policy can live — on identities, on resources, on session tokens, as boundaries, as Service Control Policies. When an actual API call comes in — Alice trying to read an object from a bucket — how does AWS combine all of those? Whose "yes" matters, and whose "no" wins?

This act is the evaluation engine. It closes with one specific safety pattern — **permission boundaries** — that lets you hand IAM administration to a developer without handing them the ability to grant themselves admin.

## Policy Evaluation Order

Starting position: **implicit deny** (everything is denied by default).

1. **Explicit deny** — checked across every applicable policy. Any `Deny` anywhere wins immediately. **Unconditional.**
2. **SCPs** — along the path from Org Root → OU → account. Must allow the action.
3. **Resource-based policies** — can grant access on their own (within the same account).
4. **Permission boundary** on the principal — must allow the action.
5. **Session policies** passed at assume time — must allow the action.
6. **Identity-based policies** on the principal — must allow the action.

**The single rule:** *the effective permission is the intersection of every layer that needs to say yes, and any explicit deny anywhere kills it.*

## Permission Boundaries — Delegated Administration

**Problem:** You want to let a developer create IAM roles for the services they own — but not give them the power to create a role with `AdministratorAccess` and then assume it.

**Solution:** Grant the developer `iam:CreateRole`, **with a condition** that every role they create must carry a specific permission boundary. The boundary caps what those roles can ever do, regardless of what permissions policy is attached.

> **Permission boundaries grant nothing on their own — they only set a ceiling.** They are the mechanism for safely delegating IAM itself.

# Act 4 — Letting outsiders in

Everything so far has happened inside one account. Real systems are messier. Your code in account A needs to read a bucket in account B. A vendor needs to run a backup tool against your infrastructure. Five hundred employees need to log into the console without you ever creating an IAM user for any of them. And a CI job running on GitHub Actions needs to deploy to your account without a long-lived access key sitting in a YAML file.

All of those are variants of the same problem — an identity from somewhere else needs in. The answer is the **Security Token Service** and the patterns built on top of it.

## STS — Security Token Service

The engine that mints **temporary credentials** when a role is assumed. Returns:

- Access key ID
- Secret access key
- Session token
- Expiration time (**15 min – 12 hours**)

### The Five `AssumeRole` Flavors

| API | Used for |
|---|---|
| **`AssumeRole`** | Standard same-account or cross-account assumption |
| **`AssumeRoleWithSAML`** | Federation from a SAML 2.0 IdP (ADFS, Okta, Ping) |
| **`AssumeRoleWithWebIdentity`** | Federation from an OIDC IdP (Cognito, Google, EKS service accounts, GitHub Actions) |
| **`GetSessionToken`** | Wrap an existing IAM user's keys with MFA |
| **`GetFederationToken`** | Custom identity broker you build yourself |

### Instance Profiles & Service Roles

- **Instance profile** wraps a role for an EC2 instance; the EC2 metadata service vends rotating creds. SDK picks them up automatically. **Never plant a long-lived access key on an EC2 box.**
- **Lambda execution role** does the same for Lambda.
- **ECS task role**, **EKS Pod Identity** — same pattern, different plumbing.

In [ ]:
# Sketch of a cross-account assume-role with confused-deputy protection.
# Requires real credentials + a real role to actually run.
#
# import boto3
#
# sts = boto3.client("sts")
# assumed = sts.assume_role(
#     RoleArn="arn:aws:iam::222222222222:role/CrossAccountReadOnly",
#     RoleSessionName="alice-readonly-session",
#     ExternalId="vendor-shared-secret-xyz",   # confused-deputy protection
#     DurationSeconds=3600,
# )
# creds = assumed["Credentials"]
#
# s3 = boto3.client(
#     "s3",
#     aws_access_key_id=creds["AccessKeyId"],
#     aws_secret_access_key=creds["SecretAccessKey"],
#     aws_session_token=creds["SessionToken"],
# )
# print(s3.list_buckets()["Buckets"])

## Cross-Account Access — Two Patterns

| Pattern | How it works | Caller's identity |
|---|---|---|
| **Role assumption** | Resource account creates a role trusting the caller's account; caller calls `sts:AssumeRole` | **Dropped** — caller becomes the role |
| **Resource policy** | Resource account names the caller's principal directly in the resource's policy | **Preserved** — caller keeps original identity end to end |

### The Confused Deputy Problem

When a third-party vendor assumes a role in your account, the vendor's role is trusted by **many** of their customers. You don't want a different customer of theirs to trick the vendor into reaching into *your* account.

| Condition key | Use when | What it asserts |
|---|---|---|
| **`sts:ExternalId`** | A third-party (vendor) assumes a role in your account | Caller must pass a unique secret value you generated |
| **`aws:SourceArn`** | An AWS *service* calls on your behalf | The service must be acting for a specific resource ARN |
| **`aws:SourceAccount`** | An AWS *service* calls on your behalf | The service must be acting for a specific account ID |

## Identity Federation

Let external identities assume IAM roles **without** ever becoming IAM users.

| Protocol | Used by | STS API |
|---|---|---|
| **SAML 2.0** | ADFS, Okta, Ping (enterprise IdPs) | `AssumeRoleWithSAML` |
| **OIDC** | Cognito, Google, EKS service accounts, GitHub Actions | `AssumeRoleWithWebIdentity` |

### IAM Identity Center (formerly AWS SSO)

The **modern, preferred** way to wire federation across an organization.

- Sits in your management account
- Connects to one source of truth: its own directory, Active Directory, or an external IdP (Okta, Entra ID)
- One sign-on portal listing every account + role each user can use
- Behind the scenes: calls `AssumeRole` and hands the user temporary creds

### Permission Sets (Identity Center's unit of authorization)

- A template bundling AWS managed + customer managed policies, optional permission boundary, optional session duration
- Define once, assign to users in specific accounts
- Identity Center provisions a hidden IAM role into each target account behind the scenes
- Update the permission set → the role updates everywhere it's assigned

> **New AWS environments today should default to Identity Center over per-account IAM users** — it eliminates the entire category of long-lived console passwords and access keys.

# Act 5 — Many accounts, one company

Most real organizations do not have one AWS account. They have dozens or hundreds — separated by environment, by team, by blast radius, by billing line. Production sits in one account, dev sits in another. The data science team gets their own account. Each acquired subsidiary brings two more.

That fleet needs to be steered from above — one bill, one set of rules, the ability to spin up a new account as code, and a way to say "no principal in this part of the tree can use any region outside the EU, ever." That is what **AWS Organizations** exists for.

## AWS Organizations

The service that turns one AWS account into a managed fleet of many.

### Tree structure

- **Root** — top of the tree
- **Organizational Units (OUs)** — logical containers, nestable
- **Accounts** — leaves

### Two account types

| Account | What it does |
|---|---|
| **Management** (formerly *master*) | Owns the org, pays all bills, only one that can attach SCPs and create member accounts. **Don't run workloads here.** |
| **Member** | Every other account in the org |

### Why use Organizations

| Benefit | What you get |
|---|---|
| **Consolidated billing** | One invoice; volume discounts and Savings Plans apply across the whole fleet |
| **Programmatic account creation** | Spin up new accounts as code (foundation of landing zones) |
| **Centralized governance** | SCPs and service delegation enforce rules from above |
| **Cross-account sharing** | Share Reserved Instances, Savings Plans, AMIs, etc. via RAM |

## Service Control Policies (SCPs)

A policy attached to the Org Root, an OU, or an account. Defines the **maximum permissions** for every principal in scope — **including the root user of member accounts.**

### Two strategies

| Strategy | Starting point | Add | Maintenance |
|---|---|---|---|
| **Deny-list** *(common)* | Built-in `FullAWSAccess` (allows everything) | Explicit `Deny` for what you want to block | Easier |
| **Allow-list** | Remove `FullAWSAccess` | Explicit `Allow` for every permitted action | Hard — every new service requires an update |

### Three things SCPs do *NOT* do

- **Do not grant permissions** — `Allow` in an SCP is meaningless unless IAM also grants the action
- **Do not apply to the management account** — always exempt (the reason you don't run workloads there)
- **Do not affect service-linked roles** — special pre-built roles AWS services create on your behalf

### Two patterns to recognize

- **Region restriction** — `Deny` any action where `aws:RequestedRegion` is not in your approved list, with `NotAction` to exempt global services (IAM, Organizations, Route 53, CloudFront) so they don't break
- **Lock-in to the org** — `Deny` on `organizations:LeaveOrganization` so member accounts can't escape governance

In [ ]:
# Region-restriction SCP — denies anything outside the approved regions,
# except global services that would break under a region check.
import json

scp = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "DenyOutsideApprovedRegions",
            "Effect": "Deny",
            "NotAction": [
                "iam:*",
                "organizations:*",
                "route53:*",
                "cloudfront:*",
                "support:*",
                "sts:*",
            ],
            "Resource": "*",
            "Condition": {
                "StringNotEquals": {
                    "aws:RequestedRegion": ["us-east-1", "eu-west-1"]
                }
            },
        }
    ],
}
print(json.dumps(scp, indent=2))

# Act 6 — Day-two hygiene

The mechanics are now all on the table. What is left is the toolkit — the things you reach for once the basics work, to keep an account from drifting.

There are four worth meeting. A service that watches for unintended openness and unused permissions. A pattern for writing policies that scale without explosion. A short list of condition keys that solve most real-world problems. And the second factor that should sit on every identity that matters.

## IAM Access Analyzer

Watches your account for unintended openness and for policies drifting from least privilege.

| Feature | What it does |
|---|---|
| **External access** | Continuously inspects resource policies (S3, IAM roles, KMS, Lambda, SQS, Secrets Manager…) and flags anything reachable from outside the account/org |
| **Unused access** | Flags IAM users, roles, permissions, and access keys that haven't been used recently |
| **Policy validation** | Checks policies you write *before* deployment — syntax, redundant statements, best-practice violations |
| **Policy generation** | Reads CloudTrail history for an identity and proposes a tighter policy based on what was actually used |

## ABAC — Tag-Based Access Control

**Attribute-Based Access Control** writes policies whose effect depends on **tags** rather than hard-coded ARNs.

Example: tag principals with `Project=Atlas`, tag resources with the same `Project=Atlas`, write one policy that allows access only when the principal's `Project` tag matches the resource's `Project` tag.

**When a new project starts, you don't write any new policies** — you tag the new resources and the new identities, and access flows automatically.

> ABAC is the scaling pattern when the number of distinct permission sets would otherwise explode.

## Useful Condition Keys

Conditions are how policies become context-aware. The high-leverage ones:

| Key | Use case |
|---|---|
| **`aws:MultiFactorAuthPresent`** | Force MFA before sensitive actions (combine with `Deny`) |
| **`aws:SourceIp`** | Restrict callers to specific public IP ranges (office, VPN) |
| **`aws:RequestedRegion`** | Region restriction |
| **`aws:PrincipalOrgID`** | Restrict resource access to principals from your org |
| **`aws:SecureTransport`** | Require HTTPS |
| **`aws:PrincipalTag`** / **`aws:ResourceTag`** | Building blocks of ABAC |
| **`sts:ExternalId`** | Confused-deputy protection (third-party vendor) |
| **`aws:SourceArn`** / **`aws:SourceAccount`** | Confused-deputy protection (AWS service calling on your behalf) |

## Multi-Factor Authentication (MFA)

Layers a second factor over a password — a stolen password alone is not enough to get in.

| Factor type | Examples |
|---|---|
| **Virtual authenticator app** | Google Authenticator, Authy, 1Password |
| **Hardware TOTP token** | Physical device displaying rotating codes |
| **FIDO security key** | YubiKey and similar |

**The simple rule:** MFA on root, MFA on every IAM user with console access, and **MFA enforced through condition keys** on the operations that hurt most when they go wrong (e.g. `s3:DeleteObject`, `iam:*`).